## 4. Lump-sum taxes and product taxes

The government has two instruments. A lump-sum tax `T` lowers income from `I` to
`I-T`. A product tax `tau_j >= 0` raises the price the consumer pays for good `j`
from `p_j` to `(1+tau_j)*p_j`, while the seller still receives `p_j`. Total revenue
is eq. 5,

$$R = T + \sum_{j=1}^{3}\tau_j p_j x_j^{\star},$$

where `x_j*` is what the consumer buys *given* the taxes. The revenue leaves the
model: the consumer gets nothing back.

### 4.1 Implementing the revenue equation

`GovernmentClass` inherits everything from `ConsumerClass`, so the whole of
sections 1-3 keeps working. Two methods do the work:

* `set_taxes(T,tau1,tau2,tau3)` writes the taxes into `par`, sets the prices the
  consumer faces to `(1+tau_j)*p_j_pre`, and sets income to `I_pre - T`. After
  this call, `solve()`, `shares()` and `quantities()` all describe the situation
  *with* taxes, without a single change to the code from sections 1-3.
* `tax_revenue(opt)` evaluates eq. 5 at the solution `opt`: it converts the nested
  shares `(s1,w)` into quantities and adds up `T` plus `tau_j*p_j_pre*x_j`.

The one thing worth being careful about is which price enters eq. 5. The `p_j` in
eq. 5 is the price the *seller* receives, i.e. the price before the tax was added
— which is why `sync_pre_tax()` stores `p1_pre, p2_pre, p3_pre, I_pre` at
construction time, and why `tax_revenue()` uses those and not `par.p1, par.p2,
par.p3`. Using the post-tax prices would count the tax rate twice and overstate
revenue by a factor `(1+tau_j)`.

`tax_revenue()` takes an already-solved `opt` as an argument. That is not
cosmetic: solving the consumer's problem is the expensive step, and in sections
4.2-4.5 we need revenue and utility from the *same* solution. Passing `opt` in
guarantees the two numbers describe the same consumer, and halves the work.

In [ ]:
# the two calibrations, exactly as in sections 1-3, but now as governments
gov = {}
gov['complements'] = GovernmentClass()                    # sigma_B = 0.40
gov['substitutes'] = GovernmentClass(par={'sigma_B':3.0}) # sigma_B = 3.00

for name,model in gov.items():
    par = model.par
    print(f'{name}: p1_pre={par.p1_pre:.4f}, p2_pre={par.p2_pre:.4f}, '
          f'p3_pre={par.p3_pre:.4f}, I_pre={par.I_pre:.4f}')

In [ ]:
# with all taxes at zero the consumer's problem is the one from section 2,
# and eq. 5 must give exactly zero
for name,model in gov.items():
    model.set_taxes()
    opt = model.solve(do_print=False)
    R = model.tax_revenue(opt)
    print(f'{name}: R={R:.10f}, u={opt.u:.6f}')
    assert np.isclose(R,0.0), 'no taxes must give no revenue'

In [ ]:
# a tax on the bus only: eq. 5 collapses to a single term, tau2*p2_pre*x2,
# which we can compute by hand and compare with what tax_revenue() returns
for name,model in gov.items():
    model.set_taxes(tau2=0.5)
    opt = model.solve(do_print=False)
    x1,x2,x3 = model.quantities(opt.s1,opt.w)
    R = model.tax_revenue(opt)
    R_hand = 0.5*model.par.p2_pre*x2
    print(f'{name}: x=({x1:.4f},{x2:.4f},{x3:.4f}), '
          f'R={R:.6f}, by hand={R_hand:.6f}')
    assert np.isclose(R,R_hand)

In [ ]:
# an accounting identity that must hold for any combination of taxes:
# the consumer spends I-T at the prices he pays, and revenue is the lump-sum
# tax plus the wedge between what he pays and what the sellers receive
for name,model in gov.items():
    model.set_taxes(T=0.5,tau1=0.2,tau2=0.5,tau3=1.0)
    opt = model.solve(do_print=False)
    par = model.par

    # a. quantities at the taxed prices
    x1,x2,x3 = model.quantities(opt.s1,opt.w)

    # b. spending at consumer prices and at seller prices
    spend_consumer = par.p1*x1 + par.p2*x2 + par.p3*x3
    spend_seller = par.p1_pre*x1 + par.p2_pre*x2 + par.p3_pre*x3

    # c. the two identities
    R = model.tax_revenue(opt)
    print(f'{name}: spending={spend_consumer:.6f}, I-T={par.I_pre-par.T:.6f}, '
          f'R={R:.6f}, T+wedge={par.T+spend_consumer-spend_seller:.6f}')
    assert np.isclose(spend_consumer,par.I_pre-par.T)
    assert np.isclose(R,par.T+spend_consumer-spend_seller)

In [ ]:
# the same rate on all three goods scales every price by (1+tau), which is the
# same as scaling income by 1/(1+tau). The consumer's real problem is unchanged,
# spending at seller prices is I/(1+tau), and revenue must be exactly
# R = tau/(1+tau)*I -- the only closed-form answer available in this model
for name,model in gov.items():
    for tau in [0.25,0.5,1.0,2.0]:
        R,u = model.revenue_and_utility(tau,goods=(1,2,3))
        R_formula = tau/(1+tau)*model.par.I_pre
        print(f'{name}, tau={tau:.2f}: R={R:.8f}, formula={R_formula:.8f}')
        assert np.isclose(R,R_formula)
    model.set_taxes()

In [ ]:
# a first look at the five instruments used in 4.2-4.5, at one arbitrary rate
cases = []
cases.append({'label':'no taxes','T':0.0,'tau1':0.0,'tau2':0.0,'tau3':0.0})
cases.append({'label':'lump-sum, T=0.20','T':0.20,'tau1':0.0,'tau2':0.0,'tau3':0.0})
cases.append({'label':'food only, tau=0.5','T':0.0,'tau1':0.5,'tau2':0.0,'tau3':0.0})
cases.append({'label':'bus only, tau=0.5','T':0.0,'tau1':0.0,'tau2':0.5,'tau3':0.0})
cases.append({'label':'train only, tau=0.5','T':0.0,'tau1':0.0,'tau2':0.0,'tau3':0.5})
cases.append({'label':'all three, tau=0.5','T':0.0,'tau1':0.5,'tau2':0.5,'tau3':0.5})

for name,model in gov.items():
    rows = []
    for case in cases:
        # a. set the taxes and solve once
        model.set_taxes(T=case['T'],tau1=case['tau1'],
                        tau2=case['tau2'],tau3=case['tau3'])
        opt = model.solve(do_print=False)

        # b. quantities and revenue from that same solution
        x1,x2,x3 = model.quantities(opt.s1,opt.w)
        R = model.tax_revenue(opt)

        # c. collect one row
        row = {}
        row['instrument'] = case['label']
        row['x1'] = x1
        row['x2'] = x2
        row['x3'] = x3
        row['R'] = R
        row['u'] = opt.u
        rows.append(row)

    print(name)
    display(pd.DataFrame(rows).round(4))

    # d. put the model back to no taxes, so later sections start clean
    model.set_taxes()

**What the checks show.** Four tests, from weakest to strongest:

1. With no taxes revenue is exactly zero, and utility is the number from section
   2 — so `GovernmentClass` really does reduce to the consumer of sections 1-3.
2. With a tax on the bus alone, eq. 5 has one term, and `tax_revenue()` returns
   exactly `tau2*p2_pre*x2` computed by hand.
3. With all four instruments switched on at once, spending at consumer prices
   equals `I-T` to machine precision — the budget constraint holds exactly,
   which is the reparametrization in `(s1,w)` doing its job — and revenue equals
   the lump-sum tax plus the wedge between what the consumer pays and what the
   sellers receive. This is the check that would fail if `tax_revenue()` used
   the post-tax prices.
4. A uniform tax on all three goods gives `R = tau/(1+tau)*I` to eight decimals,
   for both calibrations and every rate. This is the only case where the model
   has a closed-form answer, and it is worth the extra three lines: a uniform
   product tax scales all three prices by the same factor, which the consumer
   cannot substitute away from, so it is a lump-sum tax in disguise. It leaves
   the relative prices — and therefore the budget shares — untouched.

**Reading the table.** The rows already show what sections 4.2-4.5 are about.
The uniform tax raises 3.3333 in both calibrations, identical because of point 4.
The three single-good taxes raise very different amounts at the same rate, because
they hit goods of very different size and very different substitutability: food
is 54 percent of the budget and hard to substitute away from, so a 50 percent tax
on food raises about 1.85; a 50 percent tax on train tickets raises 0.98 under
complements but only 0.26 under substitutes, because the substitutes consumer
simply moves to the bus (`x3` falls from 0.95 to 0.34, while `x2` *rises* from
3.20 to 3.90). That is the first sign of what section 4.3 makes precise: the more
easily the consumer escapes a tax, the less it raises — and the tax on train
tickets in the substitutes calibration is the extreme case.

One coincidence worth noticing, so it is not mistaken for a bug: at `tau2 = 0.5`
the bus costs `1.5`, exactly the price of a train ticket. With `beta = 0.5` the
two travel goods then have identical weights *and* identical prices, so `x2 = x3`
in **both** calibrations regardless of `sigma_B` — the two rows are identical by
symmetry, not by accident.